In [1]:
from datetime import date
import polars as pl

from flights_ai_model_airport.features import get_training_flights
from flights_ai_model_airport.flights_ai_model_utils import get_latest_airport_model_path
from flights_ai_model_airport.inference import load_model, run_score_canidates, score_airport_candidates
MODEL_PATH = get_latest_airport_model_path()

target_date = date(2026,3,15)
df_flights = get_training_flights(target_date)

Loading airports from /Volumes/T2-SSD/planequery/data/interim/airports.csv


In [2]:
df = score_airport_candidates(df_flights, load_model(MODEL_PATH))
df

Loading airport model from /Volumes/T2-SSD/planequery/data/models/flights_ai_model_airport/runs/2026-07-02_18-45_bd2909a_9days/model.pkl


/Users/jonahgoode/Documents/PlaneQuery/Code/planequery-data/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


flight_id,airport_ident,score,distance_km,elevation_above_airport_ft,aircraft_category,airport_type,bearing_airport_to_first_deg,bearing_first_to_airport_deg,takeoff_track_error_deg,airport_behind_error_deg,along_track_from_airport_km,cross_track_from_airport_km,candidate_is_behind,rank
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32
"""a0003a_2026-03-15_12-59""","""KAPF""",0.999976,1.357153,342.0,3.0,2.0,44.011419,224.015584,178.615582,178.611417,-1.356757,0.032789,0.0,1
"""a0003a_2026-03-15_12-59""","""FA37""",0.000044,8.067243,341.0,3.0,1.0,308.187051,128.159061,82.75906,82.787049,1.016812,8.002906,0.0,2
"""a0003a_2026-03-15_12-59""","""KMKY""",0.000026,20.716662,345.0,3.0,1.0,333.274324,153.233287,107.833286,107.874322,-6.344445,19.721261,0.0,3
"""a0003a_2026-03-15_12-59""","""US-12873""",0.000024,36.352601,350.0,3.0,1.0,193.472826,13.435199,31.964803,31.927176,30.840582,19.245002,1.0,4
"""a0003a_2026-03-15_12-59""","""1FD4""",0.000022,35.315802,321.0,3.0,1.0,206.620101,26.549843,18.850159,18.779901,33.421702,11.410335,1.0,5
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""adf0ff_2026-03-15_01-59""","""US-2468""",0.000022,148.006378,0.0,2.0,1.0,202.844926,22.488224,144.41177,144.055068,-120.361795,86.133188,0.0,136
"""adf0ff_2026-03-15_01-59""","""US-3930""",0.000022,148.844796,286.0,2.0,1.0,138.767335,319.374705,152.474712,151.867341,-131.996599,68.787144,0.0,137
"""adf0ff_2026-03-15_01-59""","""US-2469""",0.000022,149.37861,0.0,2.0,1.0,200.321129,19.998966,146.901028,146.578865,-125.13872,81.573708,0.0,138


In [3]:
# we'll do the main and get what flight_ids did not match for airports.

1. Get flights where model failed to match
2. Look at what the features were for those flight_ids

In [4]:
from evaluation.flights.evaluation import main
outputs,df_stats = main("trad-airport-model", "sfdps", test_dates = target_date, airport_model_path = MODEL_PATH, pia_or_american_ladd_only=True)
df_comp = outputs[0][0]


Loading airport model from /Volumes/T2-SSD/planequery/data/models/flights_ai_model_airport/runs/2026-07-02_18-45_bd2909a_9days/model.pkl


/Users/jonahgoode/Documents/PlaneQuery/Code/planequery-data/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Stats for trad-airport-model:sfdps: {'true_positive': 894, 'false_positive': 4, 'false_negative': 6, 'precision': 0.9955456570155902, 'recall': 0.9933333333333333, 'f1': 0.9944382647385984, 'takeoff_airport_ident_match_pct': 0.9944071588366891, 'landing_airport_ident_match_pct': 0.9574944071588367, 'airport_ident_match_pct': 0.9541387024608501}
Stats CSV saved to /Volumes/T2-SSD/planequery/data/experiments/comparison/2026-07-03_00-36_bd2909a_flights_evaluation.csv


In [5]:
from data_engineering.flights.flight_type import add_flight_id_col
df_mismatched_flights = df_comp.filter((pl.col("match_status") == "both") & (pl.col("same_takeoff_airport_ident") == False))
df_mismatched_flights.select(["icao", "takeoff_time", "landing_time", "takeoff_airport_ident", "landing_airport_ident", "takeoff_airport_ident_df1", "landing_airport_ident_df1", "same_takeoff_airport_ident", "same_landing_airport_ident", "match_status"])
mismatched_flights_ids = add_flight_id_col(df_mismatched_flights).get_column("flight_id")

In [9]:
df_2 = df_mismatched_flights.select(["icao", "takeoff_time", "landing_time", "takeoff_airport_ident", "landing_airport_ident", "takeoff_airport_ident_df1", "landing_airport_ident_df1", "same_takeoff_airport_ident", "same_landing_airport_ident", "match_status"])
add_flight_id_col(df_2)

icao,takeoff_time,landing_time,takeoff_airport_ident,landing_airport_ident,takeoff_airport_ident_df1,landing_airport_ident_df1,same_takeoff_airport_ident,same_landing_airport_ident,match_status,flight_id
str,datetime[ms],datetime[ms],str,str,str,str,bool,bool,str,str
"""a03f20""",2026-03-15 18:00:10.220,2026-03-15 18:11:41.980,null,"""KRBW""","""KHXD""","""KOGB""",false,false,"""both""","""a03f20_2026-03-15_18-00"""
"""a0516f""",2026-03-15 17:26:39.550,2026-03-15 18:12:16.590,"""US-3265""","""XS93""","""KPEZ""","""KMFE""",false,false,"""both""","""a0516f_2026-03-15_17-26"""
"""a7f390""",2026-03-15 20:25:22.415,2026-03-15 22:36:33.130,"""KWST""","""KBLM""","""KACK""","""KBLM""",false,true,"""both""","""a7f390_2026-03-15_20-25"""
"""a8df96""",2026-03-15 16:27:07.675,2026-03-15 20:26:39.500,"""CO03""","""KBCT""","""KASE""","""KBCT""",false,true,"""both""","""a8df96_2026-03-15_16-27"""
"""ad15ab""",2026-03-15 13:50:35.310,2026-03-15 17:25:07.620,"""KWRB""","""KSEZ""","""KMCN""","""KSEZ""",false,true,"""both""","""ad15ab_2026-03-15_13-50"""


In [10]:
df.filter(pl.col("flight_id") == "a0516f_2026-03-15_17-26")

flight_id,airport_ident,score,distance_km,elevation_above_airport_ft,aircraft_category,airport_type,bearing_airport_to_first_deg,bearing_first_to_airport_deg,takeoff_track_error_deg,airport_behind_error_deg,along_track_from_airport_km,cross_track_from_airport_km,candidate_is_behind,rank
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32
"""a0516f_2026-03-15_17-26""","""US-3265""",0.998323,8.401352,2400.0,1.0,1.0,0.892688,180.893338,179.293338,179.292688,-8.400713,0.103616,0.0,1
"""a0516f_2026-03-15_17-26""","""KPEZ""",0.000473,2.763894,1970.0,1.0,1.0,210.649273,30.642266,29.042266,29.049273,2.416367,1.341745,1.0,2
"""a0516f_2026-03-15_17-26""","""US-5034""",0.000079,13.916355,2048.0,1.0,1.0,302.812529,122.754447,121.154447,121.212529,-7.199582,11.90928,0.0,3
"""a0516f_2026-03-15_17-26""","""TE26""",0.000053,24.455205,2125.0,1.0,1.0,312.768473,132.679436,131.079436,131.168473,-16.069631,18.434316,0.0,4
"""a0516f_2026-03-15_17-26""","""TE63""",0.000044,76.314459,2104.0,1.0,1.0,1.387065,181.396148,179.796148,179.787065,-76.313976,0.271517,0.0,5
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""a0516f_2026-03-15_17-26""","""TS76""",0.000022,146.602151,940.0,1.0,1.0,177.708469,357.738213,3.861787,3.891531,146.269279,9.873638,1.0,256
"""a0516f_2026-03-15_17-26""","""US-0473""",0.000022,146.679352,906.0,1.0,1.0,178.936869,358.950679,2.649321,2.663131,146.522574,6.779946,1.0,257
"""a0516f_2026-03-15_17-26""","""7TE9""",0.000021,26.287744,1993.0,1.0,1.0,246.529438,66.409389,64.809389,64.929438,11.188879,23.787696,0.0,258


In [8]:
df.filter(pl.col("flight_id").is_in(mismatched_flights_ids.implode())).filter(pl.col("rank") < 3)

flight_id,airport_ident,score,distance_km,elevation_above_airport_ft,aircraft_category,airport_type,bearing_airport_to_first_deg,bearing_first_to_airport_deg,takeoff_track_error_deg,airport_behind_error_deg,along_track_from_airport_km,cross_track_from_airport_km,candidate_is_behind,rank
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32
"""a0516f_2026-03-15_17-26""","""US-3265""",0.998323,8.401352,2400.0,1.0,1.0,0.892688,180.893338,179.293338,179.292688,-8.400713,0.103616,0.0,1
"""a0516f_2026-03-15_17-26""","""KPEZ""",0.000473,2.763894,1970.0,1.0,1.0,210.649273,30.642266,29.042266,29.049273,2.416367,1.341745,1.0,2
"""a7f390_2026-03-15_20-25""","""KWST""",0.02277,92.686805,3769.0,1.0,2.0,96.257943,276.985409,3.385402,2.657937,92.525058,5.473345,1.0,1
"""a7f390_2026-03-15_20-25""","""KGON""",0.008412,112.597832,3841.0,1.0,2.0,93.895908,274.782739,1.182733,0.295902,112.573843,2.324146,1.0,2
"""a8df96_2026-03-15_16-27""","""CO03""",0.092769,17.071993,13010.0,2.0,1.0,343.067129,163.030447,95.530447,95.567129,-1.645308,16.992525,0.0,1
"""a8df96_2026-03-15_16-27""","""KASE""",0.020215,22.189149,13630.0,2.0,2.0,337.009576,156.945664,89.445664,89.509576,0.214676,22.188111,0.0,2
"""ad15ab_2026-03-15_13-50""","""KWRB""",0.999869,4.508487,2406.0,2.0,2.0,302.453606,122.43168,99.268317,99.246391,-0.726128,4.449629,0.0,1
"""ad15ab_2026-03-15_13-50""","""KMCN""",0.999869,3.777979,2346.0,2.0,2.0,155.624345,335.633338,113.933341,113.924348,-1.532626,3.453141,0.0,2
